# portfolio_analysis — Pandas 평가/손익

- 담당: 이채우  브랜치: `leechaewoo`
- 규칙: 커밋 전 `Kernel → Restart & Clear Output`. 이 노트북은 1인 전용.


In [1]:
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from backend.services.portfolio.calculator import load_data, holdings, valuation, position_pnl, top_bottom
pf, sp = load_data(ROOT)            # 원본 컬럼(date/stock_id/price…)을 표준명으로 매핑해서 읽음
pf.head(), sp.head()

(  position_id   ticker            name  buy_price  quantity
 0   PF06_0001  S06_027  SYNTH_STOCK_27   24633.81        65
 1   PF06_0002  S06_028  SYNTH_STOCK_28   63905.46       328
 2   PF06_0003  S06_015  SYNTH_STOCK_15   63627.02       494
 3   PF06_0004  S06_009  SYNTH_STOCK_09   58730.40       400
 4   PF06_0005  S06_026  SYNTH_STOCK_26   63857.10       210,
   trade_date   ticker  close_price  volume
 0 2025-01-02  S06_001     43146.98  397321
 1 2025-01-02  S06_002     51070.67   54196
 2 2025-01-02  S06_003     24515.83  249071
 3 2025-01-02  S06_004     24981.83  136788
 4 2025-01-02  S06_005     28687.76  612855)

## 1. 데이터 이해 (컬럼, 결측, 중복)

In [2]:
print(pf.info()); print(sp.info())
print("결측:", pf.isna().sum().sum(), sp.isna().sum().sum(), " 중복:", pf.duplicated().sum(), sp.duplicated().sum())
print("포지션", len(pf), "종목", pf.ticker.nunique(), "| 시세 종목", sp.ticker.nunique(), "거래일", sp.trade_date.nunique(), sp.trade_date.min().date(), "~", sp.trade_date.max().date())
holdings(pf).sort_values("positions", ascending=False).head()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   position_id  300 non-null    str    
 1   ticker       300 non-null    str    
 2   name         300 non-null    str    
 3   buy_price    300 non-null    float64
 4   quantity     300 non-null    int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 11.8 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   trade_date   7560 non-null   datetime64[us]
 1   ticker       7560 non-null   str           
 2   close_price  7560 non-null   float64       
 3   volume       7560 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 236.4 KB
None
결측: 0 0  중복: 0 0
포지션 300 종목 30 | 시세 종목 30 거래일 252 2025-01-02 ~ 2025-12-19


,ticker,name,positions,quantity,cost_amount,buy_price
0,S06_001,SYNTH_STOCK_01,10,1764,8.734433e+07,49514.92
1,S06_002,SYNTH_STOCK_02,10,2075,1.094973e+08,52769.79
2,S06_003,SYNTH_STOCK_03,10,2829,8.308178e+07,29367.90
3,S06_004,SYNTH_STOCK_04,10,2500,6.441635e+07,25766.54
4,S06_005,SYNTH_STOCK_05,10,2045,5.966193e+07,29174.54


## 2. 종목별 평가금액 / 손익 / 손익률 / 비중 — 포지션을 종목별로 SUM (SQL [A-2]와 동일 계산)

In [3]:
val = valuation(pf, sp)
val

,ticker,name,positions,quantity,cost_amount,buy_price,close_price,eval_amount,pnl,pnl_pct,weight_pct
0,S06_023,SYNTH_STOCK_23,10,1910,4.412638e+07,23102.82,31178.69,5.955130e+07,15424914.40,34.96,1.60
1,S06_007,SYNTH_STOCK_07,10,1993,1.026945e+08,51527.61,68912.12,1.373419e+08,34647325.28,33.74,3.69
2,S06_021,SYNTH_STOCK_21,10,2378,1.295741e+08,54488.70,72404.80,1.721786e+08,42604496.64,32.88,4.62
3,S06_018,SYNTH_STOCK_18,10,3010,1.003891e+08,33351.85,43502.68,1.309431e+08,30554011.68,30.44,3.51
4,S06_006,SYNTH_STOCK_06,10,2626,1.132585e+08,43129.66,55732.73,1.463541e+08,33095670.09,29.22,3.93
5,S06_005,SYNTH_STOCK_05,10,2045,5.966193e+07,29174.54,36321.00,7.427644e+07,14614515.89,24.50,1.99
6,S06_004,SYNTH_STOCK_04,10,2500,6.441635e+07,25766.54,31664.95,7.916238e+07,14746029.98,22.89,2.12
7,S06_003,SYNTH_STOCK_03,10,2829,8.308178e+07,29367.90,35655.53,1.008695e+08,17787711.03,21.41,2.71
8,S06_002,SYNTH_STOCK_02,10,2075,1.094973e+08,52769.79,63966.53,1.327305e+08,23233233.01,21.22,3.56
9,S06_001,SYNTH_STOCK_01,10,1764,8.734433e+07,49514.92,59438.45,1.048494e+08,17505099.45,20.04,2.81


## 3. 상위/하위 3 종목

In [4]:
top3, bottom3 = val.head(3), val.tail(3)
pos = position_pnl(pf, sp)          # 포지션 단위: 가장 잘/못 산 매수 건 (SQL [A-4])
top3, bottom3, pos.head(5), pos.tail(5)

(    ticker            name  positions  quantity   cost_amount  buy_price  \
 0  S06_023  SYNTH_STOCK_23         10      1910  4.412638e+07   23102.82   
 1  S06_007  SYNTH_STOCK_07         10      1993  1.026945e+08   51527.61   
 2  S06_021  SYNTH_STOCK_21         10      2378  1.295741e+08   54488.70   
 
    close_price   eval_amount          pnl  pnl_pct  weight_pct  
 0     31178.69  5.955130e+07  15424914.40    34.96        1.60  
 1     68912.12  1.373419e+08  34647325.28    33.74        3.69  
 2     72404.80  1.721786e+08  42604496.64    32.88        4.62  ,
      ticker            name  positions  quantity   cost_amount  buy_price  \
 27  S06_012  SYNTH_STOCK_12         10      2865  1.844651e+08   64385.71   
 28  S06_015  SYNTH_STOCK_15         10      2304  1.462177e+08   63462.56   
 29  S06_014  SYNTH_STOCK_14         10      2461  6.197495e+07   25182.83   
 
     close_price   eval_amount          pnl  pnl_pct  weight_pct  
 27     47276.73  1.354478e+08 -49017222.22 

## 4. 결과 내보내기 (검증 담당이 읽는 파일)

In [5]:
out = ROOT/"outputs"; out.mkdir(exist_ok=True)
val.to_csv(out/"pandas_C_valuation.csv", index=False)
top_bottom(val).to_csv(out/"pandas_C_top_bottom.csv", index=False)
pos.to_csv(out/"pandas_C_positions.csv", index=False)